# Dynamic data practical: repeated persistence through physical time

This is the student investigation, built around a synthetic ring that fills in and re-forms. The setup and mathematical framing are supplied. You must record predictions, complete short computational steps, check intermediate objects and justify an interpretation.

We compute ordinary Rips persistence independently at each frame of a synthetic evolving point cloud. The output is a time series of diagrams and a CROCKER-style rank surface. No feature identity is asserted across frames.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Replace `frames` with a list of $n_t\times d$ arrays to apply the snapshot pipeline to another dynamic data set. Optional extensions come only after the core checkpoints agree.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
from ripser import ripser

def betti_curve(D,grid): return np.array([np.sum((D[:,0]<=a)&(a<D[:,1])) for a in grid])
def max_persistence(D):
 F=D[np.isfinite(D[:,1])]; return float(np.max(F[:,1]-F[:,0])) if len(F) else 0.

## Representation bridge: delay coordinates

A scalar signal is not yet a state-space point cloud. For lag $\tau$ and dimension $m$ construct

$$\Phi_{m,\tau}(t)=\bigl(x(t),x(t-\tau),\ldots,x(t-(m-1)\tau)\bigr).$$

The code below uses index lag rather than physical units. Predict the shape for a periodic signal. Then change only `lag` and describe what geometric organisation changes. Temporal order is not retained by the later Rips calculation unless it is stored separately.

In [ ]:
def delay_coordinates(signal, dimension, lag):
    rows = len(signal) - (dimension - 1) * lag
    return np.column_stack([
        signal[(dimension - 1 - j) * lag:(dimension - 1 - j) * lag + rows]
        for j in range(dimension)
    ])

sample_times = np.linspace(0, 8 * np.pi, 500)
signal = np.sin(sample_times)
lag = 25
embedded = delay_coordinates(signal, dimension=2, lag=lag)
plt.figure(figsize=(4, 4))
plt.plot(embedded[:, 0], embedded[:, 1], linewidth=1)
plt.gca().set_aspect('equal')
plt.xlabel(r'$x(t)$'); plt.ylabel(r'$x(t-\tau)$')
plt.show()
print('embedded shape:', embedded.shape, 'index lag:', lag)

## 1. Observe: participant checkpoint

The synthetic system moves from a coherent ring to a filled cloud and back. The samples at each frame are observations of a changing representation. The point labels are reused only to generate smooth motion; persistence does not use those labels.

In [ ]:
n=70; times=np.linspace(0,1,17); theta=np.linspace(0,2*np.pi,n,endpoint=False)
base_angle=theta+.03*RNG.normal(size=n); target_radius=np.sqrt(RNG.random(n))
frames=[]
for t in times:
 mix=np.sin(np.pi*t)**2
 radius=(1-mix)*np.ones(n)+mix*target_radius
 P=np.c_[radius*np.cos(base_angle),radius*np.sin(base_angle)]+.025*RNG.normal(size=(n,2))
 frames.append(P)
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,i in zip(axes,[0,8,16]): ax.scatter(*frames[i].T,s=12); ax.set_title(f't={times[i]:.2f}'); ax.set_aspect('equal')
plt.show()

## 2. Predict: participant checkpoint

1. When should the longest $H_1$ interval be largest?
2. At a fixed Rips threshold, when should $\beta_1$ be nonzero?
3. Does a long bar at consecutive frames establish that it is the same feature?
4. How might window length change the scientific interpretation?

## 3. Implement: participant checkpoint

Start with the three displayed frames at indices `0`, `8` and `16`. Compute their separate $H_1$ diagrams and longest persistence. Check your prediction before scaling the same operation to all frames.

Keep physical time $t$ distinct from filtration threshold $\varepsilon$.

In [ ]:
check_indices = [0, 8, 16]
for index in check_indices:
    diagram = ripser(frames[index], maxdim=1)['dgms'][1]
    print('index:', index, 'time:', times[index],
          'longest H1:', max_persistence(diagram))

# TODO: use the same expression in a list comprehension for every frame.
diagrams_by_time = []
# TODO: calculate max_persistence for every diagram and plot it against times.
maximum_by_time = np.array([])

## 4. Compare: participant checkpoint

Build the CROCKER-style array in two stages:

1. choose `grid = np.linspace(0, 1.8, 150)` and test `betti_curve` on the first diagram;
2. stack one curve per physical time, expecting shape `(17, 150)`.

Then compute radial coefficient of variation as a simpler baseline. The topology and baseline must use the same frames.

In [ ]:
grid = np.linspace(0, 1.8, 150)

# TODO: after diagrams_by_time is complete, calculate the first Betti curve.
first_curve = np.array([])
print('expected first-curve length:', len(grid), 'actual:', len(first_curve))

# TODO: stack all curves. Check that B.shape == (len(times), len(grid)).
B = np.empty((0, len(grid)))

def radial_coefficient_of_variation(points):
    radii = np.linalg.norm(points - points.mean(axis=0), axis=1)
    return radii.std() / radii.mean()

# TODO: compute radial_cv for every frame.
radial_cv = np.array([])
# TODO: display B.T above radial_cv using a shared physical-time axis.

## 5. Interpret: participant checkpoint

1. Which axis is physical time and which is filtration scale?
2. What statement can the rank surface support without feature tracking?
3. When would a sliding window blur a transition or create apparent persistence?
4. Does topology add information beyond radial variation in this synthetic example?
5. What extra maps or correspondences would be required to claim feature identity?

**† Qualification.** A smooth time series of summaries is still repeated static persistence. It is not a vineyard or vineyard module.

## Empirical investigation: a fish school enters a milling regime

The course subset contains detected positions from 160 frames of a school of 300 golden shiners. The supplied frame table also contains polarisation and collective rotation, two established non-topological baselines. Your question is whether framewise $H_1$ adds information beyond those baselines.

Work with positions first. Compare raw camera coordinates with a version centred on the school and divided by its root-mean-square radius. This single change distinguishes absolute group scale from relative spatial organisation. Use only the selected frames below until the calculation works.

**Predict before computing.** Should a milling state necessarily contain a persistent spatial loop? What alternative outcomes would be scientifically informative? Record how changing detection count could alter the diagram even if the school state did not change.

In [ ]:
from pathlib import Path
from io import StringIO
from urllib.request import urlopen
import csv

def find_course_file(name):
    candidates = [Path(name), Path('data')/name, Path('../../data')/name]
    return next((p for p in candidates if p.exists()), None)

def open_course_csv(name):
    path = find_course_file(name)
    if path is not None:
        return path.open()
    url = f'https://shannondeealgar.github.io/tda-masterclass/data/{name}'
    return StringIO(urlopen(url).read().decode('utf-8'))

with open_course_csv('golden_shiner_transition_frames.csv') as stream:
    frame_rows = list(csv.DictReader(stream))
selected = [0, 40, 80, 120, 152]
points_by_frame = {frame: [] for frame in selected}
with open_course_csv('golden_shiner_transition_observations.csv') as stream:
    for row in csv.DictReader(stream):
        frame = int(row['teaching_frame'])
        if frame in points_by_frame:
            points_by_frame[frame].append((float(row['x_px']), float(row['y_px'])))

empirical = []
for frame in selected:
    X = np.asarray(points_by_frame[frame])
    centred = X - X.mean(axis=0)
    rms_radius = np.sqrt(np.mean(np.sum(centred**2, axis=1)))
    X_relative = centred / rms_radius
    # TODO: compute the H1 diagram for X_relative with ripser.
    D = np.empty((0, 2))
    metadata = frame_rows[frame]
    empirical.append({
        'frame': frame,
        'n': len(X),
        'polarisation': float(metadata['polarisation']),
        'rotation': float(metadata['rotation']),
        'state': metadata['published_state_rule'],
        'max_H1_persistence': max_persistence(D),
    })

empirical
# TODO: plot the five relative clouds and compare max_H1_persistence with
# polarisation, rotation and detection count. Then repeat with raw coordinates.
# Interpret a disagreement; do not assume topology must reproduce the state labels.

## Expert optional subtopic: a SW1PerS-style periodicity score

Sliding windows turn a scalar signal into a point cloud. Centre each window and normalise its length so that the calculation responds to shape rather than offset or amplitude. Compare a sinusoid with a chirp, whose frequency changes through time. Before running the cell, predict which cloud should have the stronger persistent $H_1$ loop.

This is a small SW1PerS-style experiment, not a reimplementation of every calibration step in the published method.

In [ ]:
def forward_windows(signal, dimension, lag):
    rows = len(signal) - dimension * lag
    W = np.column_stack([signal[j*lag:j*lag+rows] for j in range(dimension+1)])
    W = W - W.mean(axis=1, keepdims=True)
    return W / np.maximum(np.linalg.norm(W, axis=1, keepdims=True), 1e-12)

u = np.linspace(0, 12*np.pi, 900)
periodic = np.sin(u)
chirp = np.sin(u + 0.018*u**2)
# TODO: build clouds with dimension=8 and lag=8.
periodic_cloud = np.empty((0, 9))
chirp_cloud = np.empty((0, 9))
# TODO: use every fourth row, compute each H1 diagram and compare maximum persistence.

## Expert optional subtopic: a genuine scale-by-codensity bifiltration

The next two axes are jointly ordered. For each point, use distance to its fifth nearest neighbour as a codensity score. Increasing $\delta$ admits points from less densely sampled regions; increasing Rips scale $\varepsilon$ adds simplices. Thus both directions give inclusions.

The output grid $\beta_1(\delta,\varepsilon)$ is a summary of the bifiltration, not a complete multiparameter barcode.

In [ ]:
angles = RNG.uniform(0, 2*np.pi, 90)
cloud = np.c_[np.cos(angles), np.sin(angles)] + 0.045*RNG.normal(size=(90,2))
cloud = np.vstack([cloud, RNG.uniform(-1.3, 1.3, size=(25,2))])
pairwise = np.linalg.norm(cloud[:,None,:] - cloud[None,:,:], axis=2)
codensity = np.sort(pairwise, axis=1)[:,5]
delta_grid = np.quantile(codensity, [0.35, 0.55, 0.75, 1.0])
epsilon_grid = np.linspace(0, 1.0, 80)
# TODO: for each delta retain cloud[codensity <= delta], compute H1,
# then stack its Betti curve on epsilon_grid. Expect four rows.
bifiltration_summary = np.empty((0, len(epsilon_grid)))
# TODO: display the grid and explain what increasing each axis forgets or includes.

## Optional extension: correspondence through change

The remaining cells compare a transient zigzag calculation with a controlled vineyard-module transport example. They may be skipped when the aim is snapshot persistence alone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
def rank_mod2(A):
 A=np.array(A,dtype=np.uint8,copy=True)%2; row=rank=0
 for col in range(A.shape[1]):
  piv=np.flatnonzero(A[row:,col])
  if not len(piv): continue
  q=row+piv[0]; A[[row,q]]=A[[q,row]]
  for i in range(A.shape[0]):
   if i!=row and A[i,col]: A[i]^=A[row]
  row+=1; rank+=1
  if row==A.shape[0]: break
 return rank

## Core 3. Implement: one transient zigzag class

Let $K_0$ and $K_2$ be the two-edge path $a-b-c$. Let $K_1$ be the triangular outline obtained by adding $c-a$. The valid diagram is $K_0\to K_1\leftarrow K_2$ because each path includes into the outline.

The displayed matrices are $\partial_1$ for the path and outline. Predict their ranks and $\beta_1=\dim C_1-\operatorname{rank}\partial_1$ before completing the code.

In [ ]:
d1_path = np.array([[1,0], [1,1], [0,1]], dtype=np.uint8)
d1_outline = np.array([[1,0,1], [1,1,0], [0,1,1]], dtype=np.uint8)
# TODO: compute rank_mod2 for both matrices.
# TODO: calculate beta1_path and beta1_outline.
# TODO: explain why the zigzag interval is supported only at K1.

## Core 1. Observe: participant checkpoint

Classify each question before choosing a method:

1. a distance threshold only increases;
2. interactions appear and disappear;
3. scale and density are both ordered parameters;
4. one fixed mesh carries a smoothly changing scalar field;
5. independent framewise diagrams are sufficient for regime detection.

## Core 2. Predict: participant checkpoint

Match the five questions to ordinary persistence, zigzag persistence, multiparameter persistence, vineyards or repeated persistence. State what additional structure each escalation retains and what it costs.

## Optional technical extension: vineyards and module transport

**Optional technical extension.** This section explores the vineyard distinction developed on the detailed correspondence page. It can be skipped without losing the core dynamic-method argument.

### A. An original teaching graphic for Turner Figure 1

The fixed disc is partitioned into pieces $A,B,C,D$ with

$$f_t(A)=21-t,\quad f_t(B)=14-t,\quad f_t(C)=15+t,\quad f_t(D)=t.$$

At each fixed physical time there are two $H_1$ intervals: $[14-t,21-t)$ and $[t,15+t)$. The line crossings at $t=3$ and $t=7$ change the ordering of critical values.

In [ ]:
t=np.linspace(0,10,300)
vals={'A: 21-t':21-t,'B: 14-t':14-t,'C: 15+t':15+t,'D: t':t}
fig,ax=plt.subplots(figsize=(9,5))
for label,y in vals.items(): ax.plot(t,y,label=label,lw=2)
for q in [3,7]: ax.axvline(q,color='black',ls='--',alpha=.5)
for q,color in zip([1,5,9],['tab:blue','tab:orange','tab:green']):
 ax.vlines(q,14-q,21-q,color=color,lw=6,alpha=.75)
 ax.vlines(q,q,15+q,color=color,lw=6,alpha=.75)
ax.set_xlabel('external time t'); ax.set_ylabel('filtration height a'); ax.set_title('Original teaching reconstruction of the Turner example'); ax.legend(ncol=2); plt.show()

### B. A two-generator transport calculation

Before the critical events use basis $([B],[D+B])$; afterwards use $([B],[D])$. Over $\mathbb F_2$, $[D]=[D+B]+[B]$, producing the transport matrix

$$T=\begin{pmatrix}1&1\\0&1\end{pmatrix}.$$

The off-diagonal entry records generator mixing that two visible vines do not show.

In [ ]:
T = np.array([[1, 1], [0, 1]], dtype=np.uint8)
e1 = np.array([1, 0], dtype=np.uint8)
e2 = np.array([0, 1], dtype=np.uint8)

# Checkpoint 1: predict T @ e1 and T @ e2, then compute them modulo 2.
image_e1 = None
image_e2 = None

# Checkpoint 2: calculate T @ T modulo 2 and compare with the identity.
T_squared = None

# Checkpoint 3: replace T by a permutation matrix. What kind of transport
# would that describe, and what mixing would disappear?
print('images:', image_e1, image_e2, 'T squared:', T_squared)

## Core 4. Compare: participant checkpoint

Complete the decision audit:

| Failure of the simpler method | Candidate escalation | Extra structure retained | New risk |
|---|---|---|---|
| Complexes shrink | Zigzag | Maps through insertions and deletions | Harder computation and interpretation |
| Two ordered parameters are essential | Multiparameter | Joint partial-order module | No complete barcode classification |
| Diagram points must be followed under a changing function | Vineyard | Vines in birth-death space | Matching can be unstable |
| Transport maps between time-indexed modules matter | Vineyard module | Coherent inter-module maps | More data and basis bookkeeping |

Which row matches the scientific question you actually need to answer?

## Core 5. Interpret: participant checkpoint

1. Why are external time and filtration height different parameters?
2. Why do two vines not imply two independent evolving homology classes?
3. What does the off-diagonal 1 in $T$ say in applied language?
4. When is repeated persistence the more honest method?
5. Why does multiparameter persistence generally lack a complete barcode?
6. What fixed scaffold could plausibly support a vineyard for a swarm or reservoir?

**◇ Object check.** A vineyard is a family of diagram-point trajectories. A vineyard module additionally retains coherent maps between persistence modules. The latter is strictly richer.